In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [3]:
from pathlib import Path

BASE_DIR = Path.cwd()

NEW_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_102"
OLD_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_101"

print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# SPEEDY -> ACCESS-OM2 variable names
SPEEDY_VARIABLES = {
    "U0": "uas",
    "V0": "vas",
    "MSLP": "psl",
}

# Output directory
FORCING_OUT_DIR = (BASE_DIR / ".." / ".." / ".." / "SPEEDY_access" / "access_forcing").resolve()
FORCING_OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_ONE_FILE_PER_YEAR = True

# Keep SPEEDY grid untouched until JRA-55 metadata/grid are inspected
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False

print("Output directory:", FORCING_OUT_DIR)
print("Variables:", SPEEDY_VARIABLES)

NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing
Variables: {'U0': 'uas', 'V0': 'vas', 'MSLP': 'psl'}


In [4]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

attm102.ctl


<xarray.Dataset> Size: 17GB
Dimensions:  (time: 8760, lev: 8, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lev      (lev) float64 64B 925.0 850.0 700.0 500.0 300.0 200.0 100.0 30.0
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables: (12/43)
    GH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    TEMP     (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    U        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    V        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    Q        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    RH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    ...       ...
    SHF      (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    LSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SLRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SNOW     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    comment:  geopotential height               [m]
    storage:  99
    title:    Means/variances
    undef:    9.999e+19
    pdef:     None

In [5]:
# Check SPEEDY variables

for speedy_var, access_var in SPEEDY_VARIABLES.items():

    if speedy_var not in ds:
        raise KeyError(
            f"{speedy_var!r} is absent from the SPEEDY dataset. "
            f"Available variables: {list(ds.data_vars)}"
        )

    var = ds[speedy_var]

    print(f"\n{'='*60}")
    print(f"{speedy_var} -> {access_var}")
    print(f"{'='*60}")

    print(var)
    print("Dimensions:", var.dims)
    print("Shape:", var.shape)
    print("Dtype:", var.dtype)
    print("Attributes:", var.attrs)

    print(
        f"{speedy_var} range:",
        float(var.min().compute()),
        "to",
        float(var.max().compute()),
    )

    print(
        f"{speedy_var} global mean:",
        float(var.mean(skipna=True).compute()),
    )

# Check temporal resolution only once
dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))


U0 -> uas
<xarray.DataArray 'U0' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  near-surface u-wind             [m/s]
    storage:  99
Dimensions: ('time', 'lat', 'lon')
Shape: (8760, 48, 96)
Dtype: >f4
Attributes: {'comment': 'near-surface u-wind             [m/s]', 'storage': '99'}
U0 range: -36.621543884277344 to 38.93262481689453
U0 global mean: -0.03135531768202782

V0 -> vas
<xarray.DataArray 'V0' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21

In [6]:
# =========================
# Build ACCESS-OM2 wind + pressure forcing variables

forcing_vars = {}

for speedy_var, access_var in SPEEDY_VARIABLES.items():
    var = ds[speedy_var].rename(access_var).astype("float32")

    if speedy_var == "MSLP":
        var = var * 100.0   # hPa -> Pa

    if SHIFT_LONGITUDE_TO_MINUS180_180:
        var = var.assign_coords(lon=((var.lon + 180.0) % 360.0) - 180.0).sortby("lon")

    if SORT_LATITUDE_NORTH_TO_SOUTH:
        var = var.sortby("lat", ascending=False)

    if access_var == "uas":
        var.attrs = {
            "standard_name": "eastward_wind",
            "long_name": "Near-surface eastward wind",
            "units": "m s-1",
            "cell_methods": "area: mean time: point",
            "source_variable": "U0",
            "source_model": "SPEEDY",
            "mapping_note": "SPEEDY U0 -> ACCESS-OM2 uas",
        }

    elif access_var == "vas":
        var.attrs = {
            "standard_name": "northward_wind",
            "long_name": "Near-surface northward wind",
            "units": "m s-1",
            "cell_methods": "area: mean time: point",
            "source_variable": "V0",
            "source_model": "SPEEDY",
            "mapping_note": "SPEEDY V0 -> ACCESS-OM2 vas",
        }

    elif access_var == "psl":
        var.attrs = {
            "standard_name": "air_pressure_at_mean_sea_level",
            "long_name": "Mean sea level pressure",
            "units": "Pa",
            "cell_methods": "area: mean time: point",
            "source_variable": "MSLP",
            "source_model": "SPEEDY",
            "mapping_note": "SPEEDY MSLP converted from hPa to Pa",
        }

    forcing_vars[access_var] = var

forcing_ds = xr.Dataset(forcing_vars)

# JRA55-do 3hrPt convention
time = forcing_ds.time
forcing_ds["time_bnds"] = xr.DataArray(
    np.stack([(time - np.timedelta64(90, "m")).values,
              (time + np.timedelta64(90, "m")).values], axis=1),
    dims=("time", "bnds"), coords={"time": time, "bnds": [0, 1]}
)

forcing_ds["lat"].attrs.update({"standard_name": "latitude", "long_name": "Latitude", "units": "degrees_north", "axis": "Y"})
forcing_ds["lon"].attrs.update({"standard_name": "longitude", "long_name": "Longitude", "units": "degrees_east", "axis": "X"})
forcing_ds["time"].attrs.update({"standard_name": "time", "long_name": "time", "axis": "T", "bounds": "time_bnds"})

forcing_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hrPt",
    "history": "Created from SPEEDY U0, V0 and MSLP",
    "comment": "SPEEDY U0 -> uas, V0 -> vas, MSLP -> psl; MSLP converted from hPa to Pa.",
}

forcing_ds

<xarray.Dataset> Size: 485MB
Dimensions:    (time: 8760, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    uas        (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    vas        (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    psl        (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    time_bnds  (time, bnds) datetime64[ns] 140kB 1988-12-31T22:30:00 ... 1991...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hrPt
    history:      Created from SPEEDY U0, V0 and MSLP
    comment:      SPEEDY U0 -> uas, V0 -> vas, MSLP -> psl; MSLP converted fr...

In [7]:
# =========================
# Basic validation

required_dims = ("time", "lat", "lon")

expected_units = {"uas": "m s-1", "vas": "m s-1", "psl": "Pa"}

for var_name in ["uas", "vas", "psl"]:
    var = forcing_ds[var_name]

    if var.dims != required_dims:
        raise ValueError(f"Expected {var_name} dimensions {required_dims}, got {var.dims}")

    if var.attrs.get("units") != expected_units[var_name]:
        raise ValueError(f"{var_name} must have units {expected_units[var_name]!r}, got {var.attrs.get('units')!r}")

    if var.attrs.get("cell_methods") != "area: mean time: point":
        raise ValueError(f"Unexpected {var_name} cell_methods: {var.attrs.get('cell_methods')}")

    if not np.issubdtype(var.dtype, np.floating):
        raise TypeError(f"{var_name} must be floating point, got {var.dtype}")

    invalid_count = int((~np.isfinite(var)).sum().compute())
    if invalid_count:
        raise ValueError(f"Found {invalid_count} invalid values in {var_name}")

    var_min = float(var.min(skipna=True).compute())
    var_max = float(var.max(skipna=True).compute())

    if var_name in ("uas", "vas") and (var_min < -150.0 or var_max > 150.0):
        print(f"WARNING: unusual {var_name} range: {var_min:.3f} to {var_max:.3f} m s-1")

    if var_name == "psl" and (var_min < 80000.0 or var_max > 110000.0):
        print(f"WARNING: unusual psl range: {var_min:.1f} to {var_max:.1f} Pa")

    print(f"{var_name}: {var_min:.3f} to {var_max:.3f} {expected_units[var_name]}")

# JRA55-do temporal convention
if forcing_ds.attrs.get("frequency") != "3hrPt":
    raise ValueError(f"Unexpected frequency: {forcing_ds.attrs.get('frequency')}")

if "time_bnds" not in forcing_ds:
    raise ValueError("time_bnds is missing")

dt_hours = np.diff(forcing_ds.time.values) / np.timedelta64(1, "h")
width_hours = (forcing_ds.time_bnds[:, 1] - forcing_ds.time_bnds[:, 0]).values / np.timedelta64(1, "h")
midpoints = forcing_ds.time_bnds[:, 0].values + (forcing_ds.time_bnds[:, 1].values - forcing_ds.time_bnds[:, 0].values) / 2

if not np.all(dt_hours == 3):
    raise ValueError(f"Expected 3-hourly time axis, got {np.unique(dt_hours)} h")

if not np.all(width_hours == 3):
    raise ValueError(f"time_bnds must be 3 hours wide, got {np.unique(width_hours)} h")

if not np.array_equal(midpoints, forcing_ds.time.values):
    raise ValueError("time must be the midpoint of time_bnds")

print("\nValidation passed")
print("time:", forcing_ds.time.values[0], "to", forcing_ds.time.values[-1])
print("intervals [h]:", np.unique(dt_hours))
print("grid:", forcing_ds.sizes["lat"], "x", forcing_ds.sizes["lon"])

uas: -36.622 to 38.933 m s-1
vas: -36.803 to 46.851 m s-1
psl: 94565.617 to 106295.273 Pa

Validation passed
time: 1989-01-01T00:00:00.000000000 to 1991-12-31T21:00:00.000000000
intervals [h]: [3.]
grid: 48 x 96


In [8]:
field_encoding = {
    "dtype": "float32",
    "zlib": True,
    "complevel": 4,
    "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, forcing_ds.sizes["lat"], forcing_ds.sizes["lon"]),
}

encoding = {
    **{v: field_encoding.copy() for v in ["uas", "vas", "psl"]},
    "time": {
        "dtype": "float64",
        "units": "days since 1900-01-01 00:00:00",
        "calendar": "gregorian",
        "_FillValue": None,
    },
    "time_bnds": {
        "dtype": "float64",
        "units": "days since 1900-01-01 00:00:00",
        "calendar": "gregorian",
        "_FillValue": None,
    },
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

In [10]:
written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(forcing_ds.time.dt.year.values)

    for year in years:
        yearly = forcing_ds.sel(time=str(int(year)))

        if yearly.sizes["time"] != 2920:
            raise ValueError(f"{year}: expected 2920 3-hourly records, got {yearly.sizes['time']}")

        for var_name in ["uas", "vas", "psl"]:
            var_ds = yearly[[var_name, "time_bnds"]]
            output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_{int(year)}.nc"

            var_encoding = {
                var_name: encoding[var_name],
                "time": encoding["time"],
                "time_bnds": encoding["time_bnds"],
                "lat": encoding["lat"],
                "lon": encoding["lon"],
            }

            var_ds.to_netcdf(
                output_file,
                mode="w",
                format="NETCDF4",
                engine="netcdf4",
                unlimited_dims=["time"],
                encoding=var_encoding,
            )

            written_files.append(output_file)
            print(f"Wrote {output_file.name}: {var_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

else:
    for var_name in ["uas", "vas", "psl"]:
        var_ds = forcing_ds[[var_name, "time_bnds"]]
        output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_all_years.nc"

        var_encoding = {
            var_name: encoding[var_name],
            "time": encoding["time"],
            "time_bnds": encoding["time_bnds"],
            "lat": encoding["lat"],
            "lon": encoding["lon"],
        }

        var_ds.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=var_encoding,
        )

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {var_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

written_files

Wrote uas_SPEEDY_1989.nc: 2920 records, 43.39 MB
Wrote vas_SPEEDY_1989.nc: 2920 records, 44.10 MB
Wrote psl_SPEEDY_1989.nc: 2920 records, 29.84 MB
Wrote uas_SPEEDY_1990.nc: 2920 records, 43.37 MB
Wrote vas_SPEEDY_1990.nc: 2920 records, 44.05 MB
Wrote psl_SPEEDY_1990.nc: 2920 records, 29.89 MB
Wrote uas_SPEEDY_1991.nc: 2920 records, 43.33 MB
Wrote vas_SPEEDY_1991.nc: 2920 records, 44.05 MB
Wrote psl_SPEEDY_1991.nc: 2920 records, 29.86 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1991.nc')]

In [11]:
if not written_files:
    raise RuntimeError("No NetCDF files were written")

for check_file in written_files:
    var_name = check_file.name.split("_")[0]

    with xr.open_dataset(check_file, decode_times=True) as check:
        print(f"\n{'='*60}")
        print(check_file.name)
        print(f"{'='*60}")

        print(check)
        print("\nVariable attributes:")
        print(check[var_name].attrs)

        print("\nEncoding:")
        print(check[var_name].encoding)

        print("\nTime:")
        print(check.time.values[0], "to", check.time.values[-1])

        var_min = float(check[var_name].min())
        var_max = float(check[var_name].max())
        units = check[var_name].attrs.get("units", "")

        print(f"\nRange [{units}]: {var_min:.6f} to {var_max:.6f}")

        # Compare first exported field against corresponding source field
        year = int(check.time.dt.year.values[0])
        source = forcing_ds[var_name].sel(time=str(year))
        source_first = source.isel(time=0).compute()
        output_first = check[var_name].isel(time=0).load()

        max_abs_difference = float(
            np.abs(source_first - output_first).max()
        )

        print(
            "Maximum absolute difference after NetCDF round trip:",
            max_abs_difference,
        )

        if max_abs_difference != 0.0:
            print(
                "NOTE: a tiny difference may arise from dtype or NetCDF encoding."
            )

print("\nAll output files verified")


uas_SPEEDY_1989.nc
<xarray.Dataset> Size: 54MB
Dimensions:    (time: 2920, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 23kB 1989-01-01 ... 1989-12-31T21:00:00
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float64 768B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    uas        (time, lat, lon) float32 54MB ...
    time_bnds  (time, bnds) datetime64[ns] 47kB ...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hrPt
    history:      Created from SPEEDY U0, V0 and MSLP
    comment:      SPEEDY U0 -> uas, V0 -> vas, MSLP -> psl; MSLP converted fr...

Variable attributes:
{'standard_name': 'eastward_wind', 'long_name': 'Near-surface eastward wind', 'units': 'm s-1', 'cell_methods': 'area: mean time: point', 'source_variable': 'U0', 'source_model': 'SPEEDY', 'ma